[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/photoelasticity/photoelasticity_poincare.ipynb)

# Photoelasticity on the Poincaré Sphere

This example computes the fringes of photoelasticity, the pattern a stressed transparent plate shows between polarizers. A transparent plate with a circular hole, pulled along its length, gathers stress around the rim of the hole, and stress makes the material birefringent: light polarized along one principal direction of the stress travels slower than light along the other. Between polarizers that difference shows as dark and bright fringes, the contours of the stress.

The stress is modelled as a map from a surface normal to the traction across it, `Stress = Planar <- Planar`, a sum of dyads of the radial and tangential directions, and its principal stresses and directions are `stress.eigh()`. Light's polarization is modelled as a point on the Poincaré sphere, in the same space as the plate, and the stressed plate as a rotor of that sphere, the exponential of a bivector set by a principal direction and the retardance. A polarizer passes half of one plus the inner product of its state with the light's, `(1 + (analyser | state)) / 2`. The [photoelasticity notebook](photoelasticity.ipynb) computes the same fringes with the light's electric field in spacetime.

In the notation of photoelasticity the stress-optic law reads as $N = C\,t\,(\sigma_1 - \sigma_2)/\lambda$, the fringe order from the stress-optic coefficient $C$, the thickness $t$ and the principal stress difference.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import VGA3D
from examples.animation import save_figure, show_animation
from examples.optics.photoelasticity import render

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# The plate lies across z; its positions, the normals of cuts and the tractions across them are directions
# in it, antivectors. The stress maps a normal to the traction across its cut.
Planar = ga.gatype.antivector().derive.containing("z")
Stress = ga.gatype((Planar, Planar))                         # Planar <- Planar
# Light's polarization is a unit vector on the Poincaré sphere, and the plate turns that sphere.
Polarization = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Retarder = ga.gatype.rotor()

## 1. The stress around a hole

Far from the hole the plate is pulled along x. Around a circular hole whose rim is free, the stress is Kirsch's solution: resolved along the local radial and tangential directions, it redistributes the traction so that at the rim none of it is radial and none slides along the rim. The stress map is the sum of the dyads of those directions, and its principal stresses and directions come from one eigendecomposition.

In [ ]:
# The hole's radius and the half width of the window into the plate, in mm; the pixels across the window,
# each way; and the tension pulling along x, in MPa.
radius = 10.0
extent = 30.0
n_pixels = 200
tension = 30.0

In [ ]:
# The pixel centres of a square window into the plate; the pixels inside the hole are covered when drawn.
centres = (np.arange(n_pixels) + 0.5) * (2 * extent / n_pixels) - extent
positions = mv.x.dual() * centres[None, :] + mv.y.dual() * centres[:, None]       # [rows, columns] Planar
# Averaging identity and reflection in y keeps axial traction and cancels transverse traction.
remote = tension * (Planar + (mv.y >> Planar)) / 2                                # [] Planar <- Planar
distance_squared = positions.scalar_norm_squared()                                # [rows, columns] Scalar
radial = positions.normalized()                                                   # [rows, columns] Planar
# A quarter turn in the plate, from x towards y, turns the radial direction around the hole.
tangent = (mv.xy * (-np.pi / 4)).exp() >> radial                                  # [rows, columns] Planar
radius_ratio = radius**2 / distance_squared                                       # [rows, columns] Scalar

# Resolve the remote stress along the local radial and tangential directions.
mean = remote.trace() / 2                                                         # [] Scalar
anisotropy = radial.dot(remote(radial)) - mean                                    # [rows, columns] Scalar
shear = radial.dot(remote(tangent))                                               # [rows, columns] Scalar
# Kirsch's exterior solution redistributes traction around a circular, unloaded rim.
# At the rim radius_ratio is one: both radial and sliding traction vanish.
radial_stress = (mean * (1 - radius_ratio)
                 + anisotropy * (1 - 4 * radius_ratio + 3 * radius_ratio.squared()))  # [rows, columns] Scalar
tangential_stress = (mean * (1 + radius_ratio)
                     - anisotropy * (1 + 3 * radius_ratio.squared()))             # [rows, columns] Scalar
shear_stress = shear * (1 + 2 * radius_ratio - 3 * radius_ratio.squared())        # [rows, columns] Scalar
# A direction paired with `Planar`, the space of all surface normals, reads a normal component; multiplying
# by a direction turns that reading into traction. Summing the dyads builds the stress map.
stress = (radial_stress * radial * radial.dot(Planar)
          + tangential_stress * tangent * tangent.dot(Planar)
          + shear_stress * (radial * tangent.dot(Planar) + tangent * radial.dot(Planar)))  # [rows, columns] Planar <- Planar

principal, directions = stress.eigh()                                             # [rows, columns, modes] Scalar, Planar
# The two stresses differ most strongly at the top and bottom of the hole.
difference = principal[..., 1] - principal[..., 0]                                # [rows, columns] Scalar
save_figure(render.draw_stress(positions, difference, radius, tension), "photoelastic_stress");

The colours above give the principal stress difference in units of the applied tension, reaching three at the top and bottom of the rim, over a window three hole radii from the centre each way. The square is a window into an infinite plate, not a free outer edge. [MIT's hole-in-a-plate experiment](https://ocw.mit.edu/courses/2-002-mechanics-and-materials-ii-spring-2004/09e8137971e754b1b7290a33a100bb05_lab_4_s04.pdf) gives the stress solution and its experimental context.

## 2. Retardance on the Poincaré sphere

Light polarized along one principal direction is delayed more than light along the other. Any other polarization is a mix of the two, and the growing delay between them carries it through elliptical states and back. On the Poincaré sphere, linear polarizations lie on the equator at twice their angle in the plate, so that a direction and its opposite are the same state, and the two circular polarizations sit at the poles. The sphere lives in the same space as the plate, with the plate's plane as its equator: `mv.x` is the state polarized along the plate's x direction `mv.x.dual()`, and `mv.z`, normal to the plate, a circular state. The plate turns the whole sphere about the state polarized along a principal direction, by the retardance: the phase that the principal stress difference builds up through the thickness. A polarizer passes the share of light given by Malus's law, half of one plus the inner product of its own state with the light's. Crossed linear polarizers go dark wherever a principal direction lines up with them, since their state then lies on the axis of the turn. Circular light starts at a pole, the same distance from every point on the equator, so a circular polariscope shows only the fringes of principal stress difference.

In the Jones calculus this reads as the unitary $\operatorname{diag}(e^{i\delta/2}, e^{-i\delta/2})$ in the principal frame, acting on a complex two-component field, and an analyser reading $|\langle a|\psi\rangle|^2$.

In [ ]:
# The plate's thickness and the light's wavelength, in mm; the stress-optic coefficient, an illustrative
# sensitivity in inverse MPa; and the angles of the two crossed linear polariscopes.
thickness = 1.0
wavelength = 550e-6
stress_optic = 5.5e-5
polarizer_angles = np.deg2rad([0, 22.5])
stress_phase = 2 * np.pi * stress_optic * thickness / wavelength

In [ ]:
def polarization(direction: Planar) -> Polarization:
    # The rotor from the plate's x direction to the direction turns x by twice the angle between them:
    # the sphere's equator.
    return (direction * mv.x.dual().inverse()) >> mv.x


def transmitted(state: Polarization, analyser: Polarization) -> Scalar:
    # Malus's law: one plus the inner product of the light's state with the analyser's, halved.
    return (1 + (analyser | state)) / 2


# The mean stress delays both principal directions alike and turns nothing; their difference sets the retardance.
retardance = stress_phase * difference                                            # [rows, columns] Scalar
# The axis of the turn is the state polarized along the second principal direction. The exponential of
# half the turn, a bivector, is the rotor that turns the sphere; its plane, square to the axis, holds the
# linear state an eighth of a turn from that direction, and the circular state.
eighth_turn = (mv.xy * (-np.pi / 8)).exp()                                         # [] Rotor
half_turn = (polarization(eighth_turn >> directions[..., 1]) ^ mv.z) * (retardance / 2)   # [rows, columns] Bivector
retarder = half_turn.exp()                                                        # [rows, columns] Retarder

# Two crossed linear polariscopes, turned together, and a circular one at the sphere's pole.
linear = polarization((mv.xy * (-polarizer_angles / 2)).exp() >> mv.x.dual())            # [linear views] Polarization
incident = concatenate((linear, mv.z[None]))                                      # [views] Polarization
# Each analyser passes the opposite state: without stress, every view is dark.
analyser = -incident                                                              # [views] Polarization
exits = retarder >> incident[:, None, None]                                       # [views, rows, columns] Polarization
intensities = transmitted(exits, analyser[:, None, None])                          # [views, rows, columns] Scalar
save_figure(render.draw_polariscope(positions, intensities, radius), "photoelastic_polariscopes");

## 3. Loading the plate

The tension rises to its full value and back. Scaling the load keeps the principal directions, so only the retardance grows, and in the circular polariscope the fringes crowd out from the rim as the load rises and draw back as it falls.

In [ ]:
# The frames of one load cycle, the full tension applied slowly and released, a sequence of static
# equilibria; and the duration of each frame.
n_frames = 80
duration_ms = 90
loads = (1 - np.cos(np.linspace(0, 2 * np.pi, n_frames, endpoint=False))) / 2   # [frames]

In [ ]:
# Scaling the load keeps the principal directions, so the axis of each turn stays put
# and only its angle grows: each load scales the half turn built above.
loading = (transmitted((half_turn * load).exp() >> mv.z, -mv.z) for load in loads)   # [frames] of [rows, columns] Scalar
show_animation(render.animate_polariscope(positions, loading, radius), "photoelastic_loading", duration_ms)